# CASMI 2026 — five data facts that silently break your validation

A short, reproducible EDA of the competition data with one goal: **avoid fooling yourself locally**.
Everything below is computed from `train.parquet` / `test.parquet` (plus the official RDKit wheel for the metric key).

1. **Who is in `train`** — libraries, instruments, how many spectra per structure.
2. **The visible `test.parquet` lives inside `train`** — every one of its spectra has a near-identical copy there, so a lookup scores ~1.0 and says nothing about the hidden test.
3. **The metric key is not the `inchikey14` column** — tautomer canonicalisation changes the key for a non-trivial share of structures.
4. **Test spectra are much denser than library spectra** — peak handling matters more than it looks.
5. **Adducts and collision energies** — the spellings and the summed `20,40,60` spectra you have to handle.

Each section ends with a one-line takeaway.

In [1]:
import glob, os, re, subprocess, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

def find_data():
    for pat in ('/kaggle/input/**/test.parquet', 'data/test.parquet', '../../data/test.parquet'):
        hits = [h for h in glob.glob(pat, recursive=True) if 'casmi' in h.lower() or not h.startswith('/kaggle')]
        if hits:
            return os.path.dirname(sorted(hits, key=len)[0])
    raise FileNotFoundError('attach the competition data')

DATA = find_data()
TRAIN, TEST = f'{DATA}/train.parquet', f'{DATA}/test.parquet'
pf = pq.ParquetFile(TRAIN)
print(DATA, '| train rows:', pf.metadata.num_rows, '| row groups:', pf.num_row_groups)

## 1. Who is in `train`

In [2]:
META = ['ingest_lib', 'instrument_type', 'adduct', 'ionization_mode', 'precursor_mz', 'num_peaks',
        'collision_energy_orig', 'inchikey14', 'normalized_smiles']
tr = pd.read_parquet(TRAIN, columns=META)
lib = (tr.groupby('ingest_lib')
         .agg(spectra=('inchikey14', 'size'), structures=('inchikey14', 'nunique'),
              instrument=('instrument_type', lambda s: s.value_counts().index[0]),
              median_peaks=('num_peaks', 'median'))
         .sort_values('spectra', ascending=False))
lib['spectra_per_structure'] = (lib.spectra / lib.structures).round(1)
print(f'{len(tr):,} spectra, {tr.inchikey14.nunique():,} structures (by inchikey14)')
lib

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
lib.spectra.plot.barh(ax=ax[0], log=True, title='spectra per library (log)')
lib.median_peaks.plot.barh(ax=ax[1], title='median peaks per spectrum')
for a in ax: a.invert_yaxis()
plt.tight_layout(); plt.show()

**Takeaway.** `enveda-180` (timsTOF) dominates `train` and has several spectra per structure (different collision
energies and adducts). Public libraries are much smaller per structure and much sparser. Overlap between libraries
is small, so a split by library and a split by structure are very different things.

## 2. The visible `test.parquet` lives inside `train`

For every test spectrum we look up `train` spectra of the same adduct within ±5 ppm of its precursor m/z and compute
a plain binned cosine (0.01 Da bins, √intensity, top-100 peaks). If the molecule's structure is in `train`, one of
those spectra belongs to the same compound and the cosine is close to 1.

In [4]:
te = pd.read_parquet(TEST)
print(te.shape, '| molecules:', te.molecule_id.nunique(), '| instruments:', te.instrument_type.value_counts().to_dict())

def prep(mz, it, pmz, top=100, bin_w=0.01):
    mz, it = np.asarray(mz, float), np.asarray(it, float)
    keep = mz < pmz - 0.5
    mz, it = mz[keep], it[keep]
    if len(mz) > top:
        idx = np.argsort(it)[-top:]
        mz, it = mz[idx], it[idx]
    b = np.round(mz / bin_w).astype(np.int64)
    ub, inv = np.unique(b, return_inverse=True)
    w = np.bincount(inv, weights=np.sqrt(np.clip(it, 0, None)))
    n = np.linalg.norm(w)
    return (ub, w / n) if n > 0 else None

def cosine(a, b):
    if a is None or b is None:
        return 0.0
    _, ia, ib = np.intersect1d(a[0], b[0], assume_unique=True, return_indices=True)
    return float(a[1][ia] @ b[1][ib])

# train rows inside any test window (same adduct, ±5 ppm) — filtered row group by row group to keep memory low
tp = np.sort(te.precursor_mz.to_numpy())
cols = ['ingest_lib', 'adduct', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities', 'inchikey14']
parts = []
for rg in range(pf.num_row_groups):
    pm = pf.read_row_group(rg, columns=['precursor_mz']).column(0).to_numpy()
    j = np.clip(np.searchsorted(tp, pm), 1, len(tp) - 1)
    near = np.minimum(np.abs(tp[j] - pm), np.abs(tp[j - 1] - pm)) <= pm * 5e-6
    if near.any():
        parts.append(pf.read_row_group(rg, columns=cols).to_pandas().loc[near])
cand = pd.concat(parts, ignore_index=True)
print(f'train spectra inside test windows: {len(cand):,}')

In [5]:
cand['spec'] = [prep(m, i, p) for m, i, p in zip(cand.ms2_mzs, cand.ms2_normalized_intensities, cand.precursor_mz)]
cand = cand.sort_values('precursor_mz').reset_index(drop=True)
cp = cand.precursor_mz.to_numpy()
rows = []
for r in te.itertuples():
    q = prep(r.ms2_mzs, r.ms2_normalized_intensities, r.precursor_mz)
    a, b = np.searchsorted(cp, [r.precursor_mz * (1 - 5e-6), r.precursor_mz * (1 + 5e-6)])
    sub = cand.iloc[a:b]
    sub = sub[sub.adduct == r.adduct]
    best, lib_best = 0.0, None
    for s, l in zip(sub.spec, sub.ingest_lib):
        c = cosine(q, s)
        if c > best:
            best, lib_best = c, l
    rows.append((r.molecule_id, r.collision_energy_orig, best, lib_best))
per_spec = pd.DataFrame(rows, columns=['molecule_id', 'ce', 'best_cos', 'lib'])
per_mol = per_spec.groupby('molecule_id').best_cos.max()
print(f'test spectra with an (almost) identical train spectrum, cosine ≥ 0.999: {(per_spec.best_cos >= 0.999).mean():.1%}'
      f' | ≥ 0.9: {(per_spec.best_cos >= 0.9).mean():.1%}')
print(f'test molecules with at least one cosine ≥ 0.999 match: {(per_mol >= 0.999).mean():.1%}')
print('library of the best match:', per_spec.lib.value_counts().to_dict())
print(per_spec.assign(identical=per_spec.best_cos >= 0.999).groupby('ce').identical.mean().round(3).to_dict())

**Takeaway.** Every single visible-test spectrum (1213 of 1213, at every collision energy and adduct) has a
practically identical copy in `train` (cosine ≥ 0.999, all from `enveda-180`), so a nearest-neighbour lookup scores
close to 1.0 on `test.parquet` — and that number tells you nothing about the
hidden test (the leaderboard top is far below 1.0). Validate by **holding out whole structures** (all of their spectra)
from `train` and building queries the way the test is built; also keep a mode where the answer is not in your
candidate pool at all, which must score exactly 0 (a cheap leakage check).

## 3. The metric key is not the `inchikey14` column

The metric compares `MolToInchiKey(TautomerEnumerator().Canonicalize(mol))[:14]` computed with **RDKit 2026.3.3**.
The `inchikey14` column is the plain InChIKey. They differ whenever tautomer canonicalisation moves a hydrogen.

In [6]:
try:
    import rdkit
    assert rdkit.__version__ == '2026.03.3'
except Exception:
    whl = [w for w in glob.glob('/kaggle/input/**/rdkit-2026.3.3*.whl', recursive=True)
           if f'cp{sys.version_info.major}{sys.version_info.minor}' in w]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '-q', whl[0]], check=True)
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog('rdApp.*')
print('RDKit', rdkit.__version__)

_te = rdMolStandardize.TautomerEnumerator()
def metric_key(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    try:
        return Chem.MolToInchiKey(_te.Canonicalize(m))[:14]
    except Exception:
        return None

st = tr.drop_duplicates('inchikey14')[['inchikey14', 'normalized_smiles']]
sample = st.sample(4000, random_state=0).reset_index(drop=True)
t0 = time.time()
sample['metric_key'] = [metric_key(s) for s in sample.normalized_smiles]
diff = sample[sample.metric_key.notna() & (sample.metric_key != sample.inchikey14)]
print(f'{len(sample)} random structures in {time.time() - t0:.0f}s: metric key ≠ inchikey14 for {len(diff) / len(sample):.1%}')
diff.head(8)

**Takeaway.** For a few percent of structures the metric key differs from `inchikey14`; some of those collide with
the metric key of *another* `train` structure (tautomeric duplicates). Use the metric key for labels, for
de-duplicating candidate lists (two tautomers in your top-25 waste a slot) and for train/validation splits —
otherwise the "same" molecule can sit on both sides of your split.

## 4. Test spectra are much denser than library spectra

In [7]:
te['n_peaks'] = te.ms2_mzs.map(len)
dens = pd.concat([tr[['ingest_lib', 'num_peaks']].rename(columns={'ingest_lib': 'source', 'num_peaks': 'n'}),
                  te.assign(source='visible test')[['source', 'n_peaks']].rename(columns={'n_peaks': 'n'})])
q = dens.groupby('source').n.describe(percentiles=[.5, .9])[['count', '50%', '90%', 'max']].sort_values('50%')
print(q.round(0))
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, g in dens.groupby('source'):
    if name in ('visible test', 'enveda-180') or len(g) > 200_000:
        ax.hist(np.log10(g.n.clip(lower=1)), bins=60, histtype='step', density=True, label=name)
ax.set_xlabel('log10(peaks per spectrum)'); ax.legend(); plt.show()

**Takeaway.** Test spectra carry several times more peaks than most library spectra (lots of low-intensity noise).
Similarity scores are sensitive to this: decide explicitly on an intensity floor / top-k peaks / intensity
transform and check the choice on held-out structures, not on the visible test.

## 5. Adducts and collision energies

In [8]:
print('test adducts:', te.adduct.value_counts().to_dict())
print('distinct adduct spellings in train:', tr.adduct.nunique())
print('train adducts not seen in test (top):', tr.adduct[~tr.adduct.isin(te.adduct.unique())].value_counts().head(8).to_dict())
print()
print('test collision_energy_orig:', te.collision_energy_orig.value_counts().to_dict())
summed = te.collision_energy_ev.map(lambda v: len(v) > 1)
print(f'summed multi-CE spectra in test: {summed.mean():.1%} of rows')

**Takeaway.** Make sure your neutral-mass table covers every test adduct (e.g. `[M+CH2O2-H]-`), decide what to do
with the many `train`-only spellings (`[2M+Na]+`, `[M-H2O+H]+`, …), and remember that about a quarter of the test
rows are **summed spectra** (`20,40,60`): treat them as their own collision-energy condition.

---
**Summary.** Hold out structures (by the metric key), keep a "not in pool" sanity mode at exactly zero, de-duplicate
candidates by the metric key, handle dense spectra deliberately, and cover all test adducts and the summed spectra.
If this saved you a few wasted submissions, an upvote helps others find it.